In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA gold;
SET TIME ZONE 'UTC';

In [0]:
CREATE TABLE IF NOT EXISTS tibia_analytics.gold.tmp_cohort_retention AS
WITH cohort_anchor AS (
   SELECT granularity,
          period_start AS cohort_period,
          period_status AS cohort_period_status,
          is_partial_period AS cohort_is_partial_period,
          character_id,
          world_id,
          --server_type,
          --server_region,
          account_status,
          sex,
          CASE
            WHEN vocation LIKE '%Druid%'    THEN 'Druid'
            WHEN vocation LIKE '%Knight%'   THEN 'Knight'
            WHEN vocation LIKE '%Monk%'     THEN 'Monk'
            WHEN vocation LIKE '%Paladin%'  THEN 'Paladin'
            WHEN vocation LIKE '%Sorcerer%' THEN 'Sorcerer'
            ELSE vocation
          END AS vocation,
          in_guild,
          owns_house,
          is_married,
          loyalty_title
     FROM tibia_analytics.gold.characters_behavior_periodic
    WHERE lifecycle_event = 'new'
  QUALIFY ROW_NUMBER() OVER (
            PARTITION BY granularity, character_id
                ORDER BY period_start
          ) = 1
),
cohort_base AS (
  SELECT granularity,
         cohort_period,
         cohort_period_status,
         cohort_is_partial_period,
         world_id,
         account_status,
         sex,
         vocation,
         in_guild,
         owns_house,
         is_married,
         loyalty_title,
         COUNT(DISTINCT character_id) AS cohort_size
    FROM cohort_anchor
   GROUP BY granularity,
            cohort_period,
            cohort_period_status,
            cohort_is_partial_period,
            world_id,
            account_status,
            sex,
            vocation,
            in_guild,
            owns_house,
            is_married,
            loyalty_title
),
observations AS (
  SELECT ca.granularity,
         ca.cohort_period,
         ca.cohort_period_status,
         ca.cohort_is_partial_period,
         ca.character_id,
         ca.world_id,
         -- ca.server_type,
         -- ca.server_region,
         ca.account_status,
         ca.sex,
         ca.vocation,
         ca.in_guild,
         ca.owns_house,
         ca.is_married,
         ca.loyalty_title,
         obs.period_start AS observation_period,
         obs.period_status AS observation_period_status,
         obs.is_partial_period AS observation_is_partial_period,
         obs.is_active_in_period
    FROM cohort_anchor AS ca
    LEFT JOIN tibia_analytics.gold.characters_behavior_periodic AS obs
      ON obs.character_id  = ca.character_id
     AND obs.granularity   = ca.granularity
     AND obs.period_start >= ca.cohort_period
),
retention_events AS (
  SELECT granularity,
         cohort_period,
         cohort_period_status,
         cohort_is_partial_period,
         observation_period,
         observation_period_status,
         observation_is_partial_period,
         world_id,
         account_status,
         sex,
         vocation,
         in_guild,
         owns_house,
         is_married,
         loyalty_title,
         COUNT(DISTINCT CASE WHEN is_active_in_period THEN character_id END) AS retained_count
    FROM observations
   GROUP BY granularity,
            cohort_period,
            cohort_period_status,
            cohort_is_partial_period,
            observation_period,
            observation_period_status,
            observation_is_partial_period,
            world_id,
            account_status,
            sex,
            vocation,
            in_guild,
            owns_house,
            is_married,
            loyalty_title
)
SELECT re.granularity,
       re.cohort_period,
       re.cohort_period_status,
       re.cohort_is_partial_period,
       re.observation_period,
       re.observation_period_status,
       re.observation_is_partial_period,
       -- Periods elapsed since acquisition (0 = acquisition period itself)
       CASE re.granularity
         WHEN 'Day'     THEN DATEDIFF(re.observation_period, re.cohort_period)
         WHEN 'Week'    THEN CAST(DATEDIFF(re.observation_period, re.cohort_period) / 7  AS INT)
         WHEN 'Month'   THEN CAST(MONTHS_BETWEEN(re.observation_period, re.cohort_period) AS INT)
         WHEN 'Quarter' THEN CAST(MONTHS_BETWEEN(re.observation_period, re.cohort_period) / 3 AS INT)
       END AS periods_elapsed,
       -- Cohort dimension columns (all available for dashboard filtering)
       re.world_id,
       -- re.server_type,
       -- re.server_region,
       re.account_status,
       re.sex,
       re.vocation,
       re.in_guild,
       re.owns_house,
       re.is_married,
       re.loyalty_title,
       -- Metrics
       cb.cohort_size,
       re.retained_count,
       ROUND(re.retained_count / NULLIF(cb.cohort_size, 0), 4) AS retention_rate
  FROM retention_events AS re
  LEFT JOIN cohort_base AS cb
    ON cb.granularity              = re.granularity
   AND cb.cohort_period            = re.cohort_period
   AND cb.cohort_period_status     = re.cohort_period_status
   AND cb.cohort_is_partial_period = re.cohort_is_partial_period
   AND cb.world_id                 = re.world_id
   -- AND cb.server_type              = re.server_type
   -- AND cb.server_region            = re.server_region
   AND cb.account_status           = re.account_status
   AND cb.sex                      = re.sex
   AND cb.vocation                 = re.vocation
   AND cb.in_guild                 = re.in_guild
   AND cb.owns_house               = re.owns_house
   AND cb.is_married               = re.is_married
   AND cb.loyalty_title            = re.loyalty_title

In [0]:
TRUNCATE TABLE tibia_analytics.gold.cohort_retention;

In [0]:
INSERT INTO tibia_analytics.gold.cohort_retention (
  granularity,
  cohort_period,
  cohort_period_status,
  cohort_is_partial_period,
  observation_period,
  observation_period_status,
  observation_is_partial_period,
  periods_elapsed,
  world_id,
  -- server_type,
  -- server_region,
  account_status,
  sex,
  vocation,
  in_guild,
  owns_house,
  is_married,
  loyalty_title,
  cohort_size,
  retained_count,
  retention_rate,
  processed_at
)
SELECT granularity,
       cohort_period,
       cohort_period_status,
       cohort_is_partial_period,
       observation_period,
       observation_period_status,
       observation_is_partial_period,
       periods_elapsed,
       world_id,
       -- server_type,
       -- server_region,
       account_status,
       sex,
       vocation,
       in_guild,
       owns_house,
       is_married,
       loyalty_title,
       cohort_size,
       retained_count,
       retention_rate,
       CURRENT_TIMESTAMP()
  FROM tibia_analytics.gold.tmp_cohort_retention;

In [0]:
DROP TABLE IF EXISTS tibia_analytics.gold.tmp_cohort_retention;